In [1]:
from PIL import Image,ImageOps
import numpy as np
import random

def rgb_shift(pil_img, r_shift=10, g_shift=10, b_shift=10):
    img_np = np.array(pil_img).astype(np.int16)  # 防止溢出
    img_np[..., 0] = np.clip(img_np[..., 0] + random.randint(-r_shift, r_shift), 0, 255)
    img_np[..., 1] = np.clip(img_np[..., 1] + random.randint(-g_shift, g_shift), 0, 255)
    img_np[..., 2] = np.clip(img_np[..., 2] + random.randint(-b_shift, b_shift), 0, 255)
    return Image.fromarray(img_np.astype(np.uint8))

def channel_shuffle(pil_img):
    img_np = np.array(pil_img)
    channels = [0, 1, 2]
    random.shuffle(channels)
    img_np = img_np[..., channels]
    return Image.fromarray(img_np)

def hue_saturation_value(pil_img, hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10):
    img_hsv = pil_img.convert('HSV')
    hsv_np = np.array(img_hsv).astype(np.int16)

    # 随机扰动
    h_shift = random.randint(-hue_shift_limit, hue_shift_limit)
    s_shift = random.randint(-sat_shift_limit, sat_shift_limit)
    v_shift = random.randint(-val_shift_limit, val_shift_limit)

    hsv_np[..., 0] = (hsv_np[..., 0] + h_shift) % 256  # Hue是环状的
    hsv_np[..., 1] = np.clip(hsv_np[..., 1] + s_shift, 0, 255)
    hsv_np[..., 2] = np.clip(hsv_np[..., 2] + v_shift, 0, 255)

    img_result = Image.fromarray(hsv_np.astype(np.uint8), mode='HSV').convert('RGB')
    return img_result


def apply_posterize(pil_img, bits=3):
    """
    bits: 颜色位深，范围 [1, 8]。值越小，颜色越少，图像越块状。
    """
    return ImageOps.posterize(pil_img, bits)

In [2]:
img = Image.open("./dataset/ISOD_dataset/train/DUTS-TR/RGB/ILSVRC2012_test_00000072.jpg").convert("RGB")
gt = Image.open("./dataset/ISOD_dataset/train/DUTS-TR/GT/ILSVRC2012_test_00000072.png").convert("RGB")
# 颜色扰动
img_shifted = rgb_shift(img, r_shift=20, g_shift=10, b_shift=15)

# 通道打乱
img_shuffled = channel_shuffle(img_shifted)

img_shuffled = hue_saturation_value(img)

img_shuffled = apply_posterize(img)



In [3]:
def suppress_foreground_contrast(img_rgb: Image.Image, gt_mask: Image.Image, alpha=0.6):
    """
    将前景颜色混合为背景均值，实现颜色弱化
    :param img_rgb: RGB图像 (PIL)
    :param gt_mask: 二值GT图像 (PIL)，前景为1，背景为0
    :param alpha: 融合权重，越大越接近背景
    :return: PIL.Image
    """
    img_np = np.array(img_rgb).astype(np.float32)
    mask_np = np.array(gt_mask.convert("L"))
    mask_bin = (mask_np > 127).astype(np.uint8)

    # 计算背景颜色均值
    bg_pixels = img_np[mask_bin == 0]
    if len(bg_pixels) == 0:  # 防止全是前景
        return img_rgb
    bg_mean = bg_pixels.mean(axis=0)  # [R, G, B]

    # 融合前景像素
    img_np[mask_bin == 1] = (1 - alpha) * img_np[mask_bin == 1] + alpha * bg_mean

    return Image.fromarray(np.clip(img_np, 0, 255).astype(np.uint8))

def suppress_contrast_with_overlay(img_rgb: Image.Image, gt_mask: Image.Image):
    """
    给前景和背景同时加一个灰色偏移掩膜
    :param img_rgb: RGB图像 (PIL)
    :param gt_mask: GT图像 (PIL)
    :param overlay_color: 掩膜颜色（推荐浅灰）
    :param alpha: 掩膜权重
    :return: PIL.Image
    """
    alpha = random.uniform(0,0.8)
    img_np = np.array(img_rgb).astype(np.float32)
    mask_np = np.array(gt_mask.convert("L"))
    mask_bin = (mask_np > 127).astype(np.uint8)

    # 计算背景颜色均值
    bg_pixels = img_np[mask_bin == 0]
    if len(bg_pixels) == 0:  # 防止全是前景
        return img_rgb
    bg_mean = bg_pixels.mean(axis=0)  # [R, G, B]
    overlay = np.full_like(img_np, bg_mean, dtype=np.float32)

    # 全图混合
    img_np = (1 - alpha) * img_np + alpha * overlay

    return Image.fromarray(np.clip(img_np, 0, 255).astype(np.uint8))




In [ ]:
img_suppressed = suppress_foreground_contrast(img, gt, alpha=0.8)

# 方法2：整图统一弱化
img_overlayed = suppress_contrast_with_overlay(img, gt)

#img_suppressed
img_overlayed

TypeError: suppress_contrast_with_overlay() got an unexpected keyword argument 'overlay_color'

In [3]:
import os
from omegaconf import OmegaConf

base_cfg = OmegaConf.load(os.path.join("./config","default","config.yaml"))
cfg2 = OmegaConf.load("./pretrained/configs/segswin/swin_base_patch4_window12_384_22kto1k_finetune.yaml")

base_cfg.BACKBONE = cfg2.MODEL
base_cfg.DATA.IMG_SIZE = cfg2.DATA.IMG_SIZE
print(OmegaConf.to_yaml(base_cfg))


AMP_ENABLE: true
AUG:
  AUTO_AUGMENT: rand-m9-mstd0.5-inc1
  COLOR_JITTER: 0.4
  CROP: null
CKPT_ROOT: ./pretrained/ckpt
DATA:
  BATCH_SIZE: 128
  DATASETS:
  - COME
  - DUT-RGBD
  - DUTS-TR
  - NJUNLPR
  DATA_ROOT: ./dataset/ISOD_dataset
  TASK: ISOD
  IMG_SIZE: 384
  NUM_WORKERS: 8
  PIN_MEMORY: true
  DROP_LAST: true
  TEXTURE: null
  BOUNDS: true
MODEL:
  BACKBONES: segswin-base
  MFUSION: LSF
  SFUSION: PATM-BAB
OUTPUT: ''
PRINT_FREQ: 10
TEST_GPU_ID: 1
BACKBONR: null
SAVE_FREQ: 10
TAG: None
TEST: null
THROUGHPUT_MODE: false
TRAIN:
  ACCUMULATION_STEPS: 1
  AUTO_RESUME: true
  BASE_LR: 2.0e-05
  CLIP_GRAD: 5.0
  EPOCHS: 30
  LAYER_DECAY: 1.0
  LR_SCHEDULER:
    DECAY_EPOCHS: 30
    DECAY_RATE: 0.1
    GAMMA: 0.1
    MULTISTEPS: []
    NAME: cosine
    WARMUP_PREFIX: true
  MIN_LR: 2.0e-07
  MOE:
    SAVE_MASTER: false
  OPTIMIZER:
    BETAS:
    - 0.9
    - 0.999
    EPS: 1.0e-08
    MOMENTUM: 0.9
    NAME: adamw
  START_EPOCH: 0
  USE_CHECKPOINT: false
  WARMUP_EPOCHS: 5
  WARMUP_

In [ ]:
import os
import yaml
from yacs.config import CfgNode as CN

def merge_config_from_file(config: CN, cfg_file: str):
    """
    递归地从 config.yaml 加载并合并 BASE 中的所有父配置
    """
    config.defrost()

    with open(cfg_file, 'r') as f:
        yaml_cfg = yaml.load(f, Loader=yaml.FullLoader)

    # 先处理 BASE
    base_files = yaml_cfg.get('BASE', [])
    for base_cfg in base_files:
        if base_cfg:  # 非空字符串
            base_path = os.path.join(os.path.dirname(cfg_file), base_cfg)
            merge_config_from_file(config, base_path)

    print(f"=> Merging config from {cfg_file}")
    config.merge_from_file(cfg_file)
    config.freeze()


def merge_config_from_args(config: CN, args):
    """
    将 argparse 的内容覆盖配置文件中的字段（前提：字段名一致）
    """
    config.defrost()

    def _check_args(name):
        return hasattr(args, name) and getattr(args, name) is not None

    if _check_args('batch_size'):
        config.DATA.BATCH_SIZE = args.batch_size
    if _check_args('data_path'):
        config.DATA.DATA_PATH = args.data_path
    if _check_args('zip'):
        config.DATA.ZIP_MODE = args.zip
    if _check_args('cache_mode'):
        config.DATA.CACHE_MODE = args.cache_mode
    if _check_args('pretrained'):
        config.MODEL.PRETRAINED = args.pretrained
    if _check_args('resume'):
        config.MODEL.RESUME = args.resume
    if _check_args('accumulation_steps'):
        config.TRAIN.ACCUMULATION_STEPS = args.accumulation_steps
    if _check_args('use_checkpoint'):
        config.TRAIN.USE_CHECKPOINT = args.use_checkpoint
    if _check_args('amp_opt_level'):
        print("[warning] Apex amp has been deprecated, please use pytorch amp instead!")
        if args.amp_opt_level == 'O0':
            config.AMP_ENABLE = False
    if _check_args('disable_amp'):
        config.AMP_ENABLE = False
    if _check_args('output'):
        config.OUTPUT = args.output
    if _check_args('tag'):
        config.TAG = args.tag
    if _check_args('eval'):
        config.EVAL_MODE = args.eval
    if _check_args('throughput'):
        config.THROUGHPUT_MODE = args.throughput
    if _check_args('mfusion'):
        config.MODEL.MFUSION = args.mfusion
    if _check_args('enable_amp'):
        config.ENABLE_AMP = args.enable_amp
    if _check_args('fused_window_process'):
        config.FUSED_WINDOW_PROCESS = args.fused_window_process
    if _check_args('fused_layernorm'):
        config.FUSED_LAYERNORM = args.fused_layernorm
    if _check_args('optim'):
        config.TRAIN.OPTIMIZER.NAME = args.optim

    config.freeze()


In [ ]:
# 假设你已有默认配置
from load_config import load_config
from addict import Addict
cfg = load_config('./config/default/config.yaml')

args = Addict()
args.gpu_id = 1
args.test_path = "./testpath/config.yaml"

# 1. 合并配置文件
merge_config_from_file(cfg, "./pretrained/configs/segswin/swin_base_patch4_window12_384_22kto1k_finetune.yaml")

# 2. 合并命令行参数
merge_config_from_args(cfg, args)

# 3. 查看最终配置
print(cfg)


In [ ]:
import torch

from networks.GSformer import GSformer,GDGSformer
from networks.models_config import parse_option

import cv2

from PIL import Image
import torchvision.transforms as transforms

def rgb_loader(path):
    return Image.open(path).convert('RGB')

def binary_loader(path):
    return Image.open(path).convert('L')

rgb_transform = transforms.Compose([
    transforms.Resize(( config.DATA.IMG_SIZE,  config.DATA.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
binary_transform = transforms.Compose([
    transforms.Resize(( config.DATA.IMG_SIZE,  config.DATA.IMG_SIZE)),
    transforms.ToTensor()])

gt_resize_transform = transforms.Compose([
    transforms.Resize(( config.DATA.IMG_SIZE,  config.DATA.IMG_SIZE),interpolation=transforms.InterpolationMode.NEAREST)
    ])

resize_transform = transforms.Compose([
    transforms.Resize(( config.DATA.IMG_SIZE,  config.DATA.IMG_SIZE))
    ])

def sig2map(sig,fname):
    #res = F.interpolate(sig, size=(360,640), mode='bilinear', align_corners=False)
    res = sig.sigmoid().data.cpu().numpy().squeeze()
    res = (res - res.min()) / (res.max() - res.min() + 1e-8)*255
    cv2.imwrite(fname,res)

In [ ]:
args,config = parse_option("test")


In [ ]:
#set device for test

model = GDGSformer(config)

print('NOW USING RGB BackBone:',args.backbone)

if args.test_model:
    state_dict = torch.load(args.test_model+'/ckpt/Best_mae_test.pth',map_location=torch.device('cpu'))

    model_dict = {}

    for k,v in state_dict.items():
        if k.startswith('module'):
            model_dict[k[7:]] = v
        else:
            model_dict[k] = v
    
    msg = model.load_state_dict(model_dict, strict=False)
    print('Pretrained weights found at {} and loaded with msg: {}'.format(args.test_model+'/ckpt/Best_mae_test.pth', msg))

else:
    print("test_model=path/to/model.pth")
    #exit(0)

model.eval()


In [ ]:
image = rgb_loader('./dataset/RGBD_dataset/train/COME/RGB/COME_Train_4.jpg')
gray = binary_loader('./dataset/RGBD_dataset/train/COME/RGB/COME_Train_4.jpg')
depth = binary_loader('./dataset/RGBD_dataset/train/COME/depth/COME_Train_4.png')
gt = rgb_loader('./dataset/RGBD_dataset/train/COME/GT/COME_Train_4.png')
namlab = rgb_loader('./dataset/RGBD_dataset/train/COME/namlab40/COME_Train_4.png')
bound = rgb_loader('./dataset/RGBD_dataset/train/COME/bound/COME_Train_4.png')

resize_transform(image).save("./fork/RGB.png")
resize_transform(depth).save("./fork/depth.png")
gt_resize_transform(gt).save("./fork/GT.png")
resize_transform(namlab).save("./fork/namlab.png")
resize_transform(bound).save('./fork/bound.png')

image = rgb_transform(image)
depth = binary_transform(depth)
gray = binary_transform(gray)




with torch.no_grad():

    image = image.unsqueeze(0)
    depth = depth.unsqueeze(0)
    gray = gray.unsqueeze(0)

    s1,s2,s3,s4,edge_sod,edge_rgb,edge_depth = model(image, torch.cat((depth,gray),dim=1))
    
    
    print(s1.shape,s2.shape,s3.shape,s4.shape)
    # sig2map(s1,"./fork/s1.png")
    # sig2map(s2,"./fork/s2.png")
    # sig2map(s3,"./fork/s3.png")
    # sig2map(s4,"./fork/s4.png")
    # sig2map(edge_sod,"./fork/edge_sod.png")
    # sig2map(edge_rgb,"./fork/edge_rgb.png")
    # sig2map(edge_depth,"./fork/edge_depth.png")


print('Test Done!')